# Notebook 02: Fine-Tuning the Model

Following the project requirements, I am going to fine-tune a pre-trained ResNet18 model on my plant disease dataset. I will freeze the base layers and only train the final classification head to recognize my specific classes.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Quick Data Setup
data_dir = '../data/raw/PlantVillage'

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
train_dataset = datasets.ImageFolder(data_dir, transform=train_transforms)
num_classes = len(train_dataset.classes)

# Taking a tiny subset just to prove the pipeline works fast on CPU
torch.manual_seed(42)
indices = torch.randperm(len(train_dataset)).tolist()
train_data = Subset(train_dataset, indices[:500]) # Just 500 images for a quick test
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

print(f"Data ready. Total classes to predict: {num_classes}")

Using device: cpu
Data ready. Total classes to predict: 16


### Step 1: Loading ResNet and Modifying the Classifier

Here is where the actual "fine-tuning" happens. I load the pre-trained ResNet18, freeze its existing knowledge, and replace the final layer to match the number of classes in my dataset.

In [2]:
# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze the base parameters
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer for fine-tuning
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)
model = model.to(device)

# Define the Loss function and Optimizer (only optimizing the new layer)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Model architecture ready for fine-tuning.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\UsEr/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [30:11<00:00, 25.9kB/s]  

Model architecture ready for fine-tuning.


### Step 2: Running the Fine-Tuning Loop

I will run this for 1 epoch just to verify the pipeline works and saves the weights correctly.

In [3]:
print("Starting fine-tuning...")

# Training Phase (1 Epoch)
model.train()
running_loss = 0.0
correct_train = 0
total_train = 0

start_time = time.time()

for inputs, labels in train_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    
    loss.backward()
    optimizer.step()
    
    running_loss += loss.item()
    _, predicted = torch.max(outputs.data, 1)
    total_train += labels.size(0)
    correct_train += (predicted == labels).sum().item()

train_acc = 100 * correct_train / total_train
epoch_mins = (time.time() - start_time) / 60

print(f"Fine-Tuning completed in {epoch_mins:.2f} mins")
print(f"Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")

# Save the trained model weights to our app folder!
import os
os.makedirs('../app', exist_ok=True)
torch.save(model.state_dict(), '../app/plant_disease_model.pth')
print("Model saved to app/plant_disease_model.pth!")

Starting fine-tuning...
Fine-Tuning completed in 0.51 mins
Train Loss: 2.2191 | Train Acc: 43.00%
Model saved to app/plant_disease_model.pth!
